# Validate primary keys
Reads the primary key registry and checks key uniqueness and null key values for every active table.

In [0]:
# Reusable primary key validation: returns metrics and a PASS/FAIL result
from pyspark.sql import functions as F

def validate_primary_key(table_name, key_columns):
    df = spark.table(table_name)

    null_condition = F.lit(False)
    for column_name in key_columns:
        null_condition = null_condition | F.col(column_name).isNull()

    metrics = df.agg(
        F.count("*").alias("row_count"),
        F.countDistinct(*key_columns).alias("distinct_key_count"),
        F.coalesce(
            F.sum(F.when(null_condition, 1).otherwise(0)), F.lit(0)
        ).alias("null_key_count")
    ).first()

    row_count = metrics["row_count"]
    distinct_key_count = metrics["distinct_key_count"]
    null_key_count = metrics["null_key_count"]
    duplicate_count = row_count - distinct_key_count - null_key_count

    test_result = "PASS" if null_key_count == 0 and duplicate_count == 0 else "FAIL"

    return {
        "table_name": table_name,
        "key_columns": key_columns,
        "row_count": row_count,
        "distinct_key_count": distinct_key_count,
        "null_key_count": null_key_count,
        "duplicate_count": duplicate_count,
        "test_result": test_result,
        "error_message": None
    }

In [0]:
# Validate every active registered table; record errors and continue
import pandas as pd

registry = (spark.table("validation_sweep.metadata.pk_registry")
                 .where("is_active")
                 .collect())

results = []

for r in registry:
    full_name = f"{r.catalog_name}.{r.schema_name}.{r.table_name}"
    try:
        results.append(validate_primary_key(full_name, list(r.key_columns)))
    except Exception as e:
        results.append({
            "table_name": full_name,
            "key_columns": list(r.key_columns),
            "test_result": "ERROR",
            "error_message": str(e).split("\n")[0][:500]
        })

columns = ["table_name", "key_columns", "test_result", 
           "row_count", "distinct_key_count", "null_key_count", "duplicate_count", 
           "error_message"]

display(pd.DataFrame(results, columns=columns))

In [0]:
# Append this run's results to the central results table
import uuid
import datetime

results_table = "validation_sweep.results.validation_results"
method = "python_native"
test_type = "pk_integrity"
run_id = str(uuid.uuid4())
run_ts = datetime.datetime.now(datetime.timezone.utc)

run_info = {
    "run_id": run_id,
    "run_ts": run_ts,
    "run_date": run_ts.date(),
    "method": method,
    "test_type": test_type,
}

target_schema = spark.table(results_table).schema
column_names = target_schema.fieldNames()
rows = []

for r in results:
    key_columns_str = ",".join(str(c) for c in r["key_columns"])
    check_id = "|".join([method, test_type, str(r["table_name"]), key_columns_str])

    row = {**r, **run_info, "check_id": check_id}
    rows.append(tuple(row.get(c) for c in column_names))

if rows:
    results_df = spark.createDataFrame(rows, schema=target_schema)
    results_df.write.mode("append").saveAsTable(results_table)
    print(f"Wrote {len(rows)} results for run {run_id}")
else:
    print("No validation results to write")

In [0]:
# Show all checks from the latest run
display(spark.table("validation_sweep.results.latest_validation_results"))

# Show the most recent history, newest first
display(spark.table("validation_sweep.results.validation_results")
        .orderBy("run_ts", ascending=False)
        .limit(50))
